In [ ]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save engineered train/test as CSV
train_eng = X_train.copy(); train_eng["Churn"] = y_train.values
test_eng  = X_test.copy();  test_eng["Churn"]  = y_test.values
train_eng.to_csv("../data/processed/train.csv", index=False)
test_eng.to_csv( "../data/processed/test.csv",  index=False)

# Save the feature matrix (preprocessed arrays) as CSVs
pd.DataFrame(X_train_processed).to_csv("../data/processed/feature_matrix_train.csv", index=False)
pd.DataFrame(X_test_processed).to_csv( "../data/processed/feature_matrix_test.csv",  index=False)

# Save the preprocessor
joblib.dump(preprocessor, "../models/preprocessor.joblib")
print("✅ Saved:")
print("   data/processed/train.csv")
print("   data/processed/test.csv")
print("   data/processed/feature_matrix_train.csv")
print("   data/processed/feature_matrix_test.csv")
print("   models/preprocessor.joblib")

print(f"\n── Final summary ──────────────────────────────────")
print(f"  Engineered features  : {X_train.shape[1]}")
print(f"  Final feature matrix : {X_train_processed.shape[1]} (after OHE)")
print(f"  Train samples        : {X_train_processed.shape[0]:,}")
print(f"  Test  samples        : {X_test_processed.shape[0]:,}")
print(f"  Train churn rate     : {y_train.mean():.1%}")
print(f"  Test  churn rate     : {y_test.mean():.1%}")
print(f"  Train samples (SMOTE): {X_train_smote.shape[0]:,} (balanced 50/50)")
print("──────────────────────────────────────────────────")

---
## Section 5 — Save Processed Data

In [ ]:
from imblearn.over_sampling import SMOTE

# Class distribution before SMOTE
print("Class distribution BEFORE SMOTE:")
print(pd.Series(y_train).value_counts().rename({0: "No Churn", 1: "Churn"}))

# Apply SMOTE — TRAINING DATA ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print("\nClass distribution AFTER SMOTE (training only):")
print(pd.Series(y_train_smote).value_counts().rename({0: "No Churn", 1: "Churn"}))
print(f"\nTraining set grew from {len(y_train):,} → {len(y_train_smote):,} samples")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (y_data, title) in zip(axes, [
    (y_train,       "Before SMOTE"),
    (y_train_smote, "After SMOTE (train only)"),
]):
    counts = pd.Series(y_data).value_counts()
    bars = ax.bar(["No Churn", "Churn"], counts.values,
                  color=sns.color_palette("Set2")[:2], edgecolor="white")
    for bar, cnt in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f"{cnt:,}", ha="center", fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Count")
plt.suptitle("SMOTE Effect on Class Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_10_smote_effect.png", bbox_inches="tight")
plt.show()

---
## Section 4 — Class Imbalance Strategy

We have three options for handling the 73/27 imbalance:

| Strategy | How it works | When to use |
|----------|-------------|-------------|
| `class_weight='balanced'` | Penalise minority class errors more | Simple, fast, built into most sklearn models |
| **SMOTE** | Synthesise new minority samples in feature space | When model still underperforms on minority class |
| Threshold tuning | Move decision boundary below 0.5 | Post-training, trades precision for recall |

> **Rule:** We NEVER apply SMOTE to test data. SMOTE is a training-only technique.  
> Applying it to test data would create fake test examples and **massively inflate** our evaluation metrics.

In [ ]:
# Train/test split on the engineered dataframe
X_train, X_test, y_train, y_test = split_data(df_eng)

# Identify column types for the preprocessor
numerical_cols = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "monthly_tenure_ratio", "total_services", "avg_monthly_spend",
    "contract_risk",
]
# Binary int flags — passed through as-is (remainder)
binary_flag_cols = [
    "has_security_bundle", "has_support", "has_streaming_bundle",
    "is_new_customer", "is_high_value",
]
# True categorical columns (to be one-hot encoded)
categorical_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
    "tenure_group",
]

preprocessor = create_preprocessor(numerical_cols, categorical_cols)

# Fit ONLY on training data — CRITICAL
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)    # only transform, never fit

print(f"Feature matrix shape:")
print(f"  Train: {X_train_processed.shape}")
print(f"  Test:  {X_test_processed.shape}")
print(f"\nOriginal features: {X_train.shape[1]}")
print(f"After OHE:          {X_train_processed.shape[1]} (OHE expanded categoricals)")

---
## Section 3 — Preprocessing Pipeline

> **Critical concept: data leakage prevention.**  
> We fit the scaler and encoder ONLY on training data.  
> If we fitted on the entire dataset, the model would have "seen" test set statistics during training — inflating metrics and giving a false picture of real-world performance.  
> `sklearn` Pipelines make this automatic and foolproof.

In [ ]:
# Point-biserial correlation of all numerical/ordinal features with Churn
num_df = df_eng.select_dtypes(include=[np.number])
corr_with_churn = num_df.corr()["Churn"].drop("Churn").sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in corr_with_churn.values]
bars = ax.barh(corr_with_churn.index[::-1], corr_with_churn.values[::-1],
               color=colors[::-1], edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
for bar, val in zip(bars, corr_with_churn.values[::-1]):
    x_pos = val + 0.005 if val >= 0 else val - 0.005
    ha = "left" if val >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", ha=ha, fontsize=9)
ax.set_xlabel("Pearson Correlation with Churn")
ax.set_title("Feature Correlation with Target (Churn)\nRed = positive, Blue = negative",
             fontweight="bold")
ax.set_xlim(corr_with_churn.min() - 0.05, corr_with_churn.max() + 0.08)
plt.tight_layout()
plt.savefig("../reports/fig_09_feature_correlations.png", bbox_inches="tight")
plt.show()

print("\nTop 10 features most correlated with Churn:")
print(corr_with_churn.head(10).to_string())

---
## Section 2 — Feature Correlation with Target

In [ ]:
# Distribution of new numerical engineered features
num_new = ["monthly_tenure_ratio", "total_services", "avg_monthly_spend",
           "contract_risk", "is_new_customer", "is_high_value"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
palette = {0: sns.color_palette("Set2")[0], 1: sns.color_palette("Set2")[1]}

for ax, col in zip(axes, num_new):
    for churn_val, color in palette.items():
        subset = df_eng[df_eng["Churn"] == churn_val][col]
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label=f"{'Churn' if churn_val else 'No Churn'}", edgecolor="white")
    ax.set_title(f"{col}", fontweight="bold")
    ax.set_xlabel(col); ax.set_ylabel("Count"); ax.legend(fontsize=8)

plt.suptitle("Engineered Features — Distribution by Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_08_engineered_features.png", bbox_inches="tight")
plt.show()

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_raw   = load_data(RAW_PATH)
df_clean = clean_data(df_raw)

print(f"Before engineering: {df_clean.shape}")
df_eng = engineer_features(df_clean)
print(f"After  engineering: {df_eng.shape}")

new_cols = [c for c in df_eng.columns if c not in df_clean.columns]
print(f"\n10 new engineered features:")
for i, c in enumerate(new_cols, 1):
    print(f"  {i:2d}. {c}")

df_eng[new_cols].head(10)

---
## Section 1 — Apply Feature Engineering

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.preprocessing import load_data, clean_data, split_data
from src.features import engineer_features, create_preprocessor, get_feature_names

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

print("✅ Imports OK")

# 02 — Feature Engineering
## ML Classification & Prediction System | Telco Customer Churn

**Goal:** Turn raw cleaned data into a rich, ML-ready feature matrix.

Good features matter MORE than model choice — a logistic regression with great features often beats a random forest with raw features.

**Sections:**
1. Apply feature engineering (10 new features)
2. Feature correlation with target
3. Preprocessing pipeline (scaling + encoding, no leakage)
4. Class imbalance strategy (SMOTE)
5. Save processed data